In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.optim as optim

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
device

device(type='cuda')

In [ ]:
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
trainDataset = datasets.MNIST(root="", train=True, download=True, transform=transform)
trainDataloader = DataLoader(trainDataset, batch_size=64, shuffle=True)

In [ ]:
testDataset = datasets.MNIST(root="", train=False, download=True, transform=transform)
testDataloader = DataLoader(testDataset, batch_size=64, shuffle=True)

In [ ]:
from torch.nn.modules.activation import ReLU
class CNNModel(nn.Module):
  def __init__(self, outputSize):
    super().__init__()

    self.Model = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Flatten(),

        nn.Linear(256 , 128),
        nn.ReLU(),

        nn.Linear(128, outputSize),
    )
  def forward(self, x):
    return self.Model(x)

In [ ]:
cnnModel = CNNModel(len(trainDataset.classes)).to(device)

In [ ]:
cnnOptimizer = optim.Adam(cnnModel.parameters(), lr=0.001)
cnnLoss = nn.CrossEntropyLoss()

In [ ]:
epochs = 10

In [ ]:
for epoch in range(epochs):
  totalLoss = 0.0
  for image, label in trainDataloader:

    image = image.to(device)
    label = label.to(device)

    op = cnnModel(image)

    loss = cnnLoss(op, label)

    totalLoss += loss.item()

    cnnOptimizer.zero_grad()
    loss.backward()
    cnnOptimizer.step()

  print(f"epochs: {epoch+1}/{epochs}, avgLoss: {(totalLoss/len(trainDataloader)):.4}")

epochs: 1/10, avgLoss: 0.1944
epochs: 2/10, avgLoss: 0.04878
epochs: 3/10, avgLoss: 0.03408
epochs: 4/10, avgLoss: 0.0287
epochs: 5/10, avgLoss: 0.02147
epochs: 6/10, avgLoss: 0.01731
epochs: 7/10, avgLoss: 0.01669
epochs: 8/10, avgLoss: 0.01339
epochs: 9/10, avgLoss: 0.01179
epochs: 10/10, avgLoss: 0.009819


In [ ]:
with torch.no_grad():
  totalLoss = 0.0
  correct = 0

  for img, label in testDataloader:
    img = img.to(device)
    label = label.to(device)

    op = cnnModel(img)
    pred = torch.argmax(op, dim=1)

    correct += (pred == label).sum().item()

  print(f"accuracy: {(correct/len(testDataset)):.2%}")

accuracy: 99.31%
